---
format:
  html:
    code-fold: true
jupyter: python3
---

## **Cell 1: Dataset Justification**

#### Dataset Selection
For this assignment, I selected the **NASA Nearest Earth Objects** dataset from Kaggle  
([https://www.kaggle.com/datasets/sameepvani/nasa-nearest-earth-objects](https://www.kaggle.com/datasets/sameepvani/nasa-nearest-earth-objects)).  
This dataset contains information about asteroids that approach Earth, including physical characteristics such as estimated diameter, relative velocity, miss distance, and a binary indicator specifying whether an asteroid is potentially hazardous.  

This dataset is well-suited for **classification tasks** since the target variable, `is_hazardous`, represents a clear binary outcome (`True` or `False`). The dataset is under 100 MB in size, is structured as a CSV file, and contains a mix of numerical and categorical attributes that make it ideal for applying **K-Nearest Neighbors (KNN)** and **Logistic Regression (LR)** models.

#### Preprocessing Plan
To prepare the dataset for modeling, I will apply the following preprocessing steps:

1. **Data Cleaning:**  
   - Remove irrelevant columns such as IDs or names that do not contribute predictive value.  
   - Handle any missing values by either imputing them (mean/median for numerical features) or removing rows if necessary.

2. **Feature Encoding:**  
   - Convert categorical features (e.g., orbiting body) into numerical form using **one-hot encoding**, ensuring that algorithms like Logistic Regression and KNN can process them effectively.

3. **Feature Scaling:**  
   - Apply **StandardScaler** to normalize numerical features so that all attributes contribute equally to distance-based methods like KNN and prevent bias toward features with larger ranges.

4. **Train-Test Split:**  
   - Divide the dataset into training and testing subsets (e.g., 80/20 split) using **stratified sampling** to maintain the proportion of hazardous vs. non-hazardous asteroids.

5. **Feature Inspection:**  
   - Examine feature correlations and distributions to detect redundancy or highly correlated features that may later be pruned.

These preprocessing steps will ensure the dataset is clean, balanced, and properly scaled for training robust and comparable classification models.

In [30]:
# Cell 2: Data Loading and Preprocessing
# --------------------------------------
# This cell downloads and loads the NASA Near-Earth Objects dataset.
# It first tries the Kaggle API (if credentials are present),
# and otherwise falls back to a public link or a locally uploaded CSV.

import os
import zipfile
import pandas as pd

# --- Optional: Install Kaggle CLI (for Colab use) ---
!pip install -q kaggle

data_dir = "/content/data"
os.makedirs(data_dir, exist_ok=True)

kaggle_dataset = "sameepvani/nasa-nearest-earth-objects"
local_csv = os.path.join(data_dir, "neo.csv")

try:
    # Check if Kaggle credentials are available
    have_keys = bool(os.environ.get("KAGGLE_USERNAME")) and bool(os.environ.get("KAGGLE_KEY"))

    if have_keys:
        print("Kaggle credentials detected. Downloading dataset via Kaggle API...")
        !kaggle datasets download -d {kaggle_dataset} -p {data_dir} --quiet
        zip_path = os.path.join(data_dir, "nasa-nearest-earth-objects.zip")
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(data_dir)
        file_path = local_csv
    else:
        # Fallback option: dataset must be available locally or via public link
        print("No Kaggle credentials found.")
        if os.path.exists(local_csv):
            print("Found local copy of dataset.")
            file_path = local_csv
        else:
            # Optionally, use opendatasets to fetch the public file without API key
            print("Attempting public download using opendatasets...")
            !pip install -q opendatasets
            import opendatasets as od
            od.download(f"https://www.kaggle.com/datasets/{kaggle_dataset}", data_dir)
            file_path = local_csv

except Exception as e:
    raise RuntimeError(
        "Dataset download failed. Ensure the CSV is present in /content/data or provide Kaggle credentials."
    ) from e

# --- Load dataset ---
df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print(f"Shape: {df.shape}")
print("Columns:", list(df.columns))

# --- Identify target column ---
possible_targets = ["is_hazardous", "is_potentially_hazardous_asteroid", "hazardous"]
target_col = None
for t in possible_targets:
    if t in df.columns:
        target_col = t
        break
if target_col is None:
    raise ValueError(f"Could not find a valid target column among {possible_targets}")

# --- Drop non-predictive identifier columns ---
drop_cols = ["id", "name", "sentry_object", "orbiting_body"]  # These columns do not provide predictive value
df = df.drop(columns=drop_cols, errors="ignore")

# --- Prepare target variable ---
y_raw = df[target_col]
y = y_raw.map({
    True: 1, False: 0, "True": 1, "False": 0,
    "Y": 1, "N": 0, 1: 1, 0: 0
}).astype(int)
X = df.drop(columns=[target_col])

# --- Identify numeric and categorical columns ---
cat_cols = [c for c in X.columns if X[c].dtype == "object"]
num_cols = [c for c in X.columns if c not in cat_cols]

# --- Define preprocessing pipelines ---
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

# --- Train/Test Split (Stratified to maintain class balance) ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- Fit and transform datasets ---
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

# --- Get feature names after preprocessing ---
feature_names = []
if len(num_cols) > 0:
    feature_names.extend(num_cols)
if len(cat_cols) > 0:
    ohe = preprocessor.named_transformers_["cat"].named_steps["onehot"]
    feature_names.extend(ohe.get_feature_names_out(cat_cols).tolist())
if not feature_names:
    feature_names = [f"feature_{i:02d}" for i in range(X_train_proc.shape[1])]

# --- Output required information ---
print("\n=== FEATURES (post-preprocessing) ===")
print(feature_names[:50] if len(feature_names) > 50 else feature_names)
if len(feature_names) > 50:
    print(f"... ({len(feature_names)} total)")

print("\n=== SHAPES ===")
print(f"X_train: {X_train_proc.shape}")
print(f"X_test:  {X_test_proc.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

print("\nPreprocessing completed successfully!")

Kaggle credentials detected. Downloading dataset via Kaggle API...
Dataset URL: https://www.kaggle.com/datasets/sameepvani/nasa-nearest-earth-objects
License(s): CC0-1.0
Dataset loaded successfully.
Shape: (90836, 10)
Columns: ['id', 'name', 'est_diameter_min', 'est_diameter_max', 'relative_velocity', 'miss_distance', 'orbiting_body', 'sentry_object', 'absolute_magnitude', 'hazardous']

=== FEATURES (post-preprocessing) ===
['est_diameter_min', 'est_diameter_max', 'relative_velocity', 'miss_distance', 'absolute_magnitude']

=== SHAPES ===
X_train: (72668, 5)
X_test:  (18168, 5)
y_train: (72668,)
y_test:  (18168,)

Preprocessing completed successfully!


## **Cell 3: Model Training Plan**

**Goal.** Train and tune two baseline classifiers — **K-Nearest Neighbors (KNN)** and **Logistic Regression (LR)** — using **only the training split** produced in Cell 2. The held-out test split will be used **once** at the end for final evaluation.

#### Hyperparameter Tuning
I will use **`sklearn.model_selection.GridSearchCV`** with **stratified 5-fold cross-validation** and **accuracy** as the scoring metric. `GridSearchCV` will search the specified grids on **`X_train_proc, y_train`** (the preprocessed training data from Cell 2) and, with `refit=True`, automatically retrain the best configuration on the **full training set**.

#### Models (scikit-learn)
- **KNN:** `sklearn.neighbors.KNeighborsClassifier`
- **Logistic Regression:** `sklearn.linear_model.LogisticRegression`

#### Hyperparameter Ranges (and rationale)
- **KNN**
  - `n_neighbors`: **odd values** in `{1, 3, 5, …, 31}` to reduce tie risks in binary classification.
  - `weights`: `{"uniform", "distance"}` to allow distance-weighted voting.
  - `p`: `{1, 2}` to evaluate Manhattan (`p=1`) vs. Euclidean (`p=2`) distances with the Minkowski metric.
  - *(Features are already scaled in Cell 2, which is important for distance-based KNN.)*

- **Logistic Regression**
  - `C`: `{0.01, 0.1, 1, 10, 100}` (log-scale covering strong to weak regularization).
  - `penalty`: `{"l2"}` with solver `lbfgs` (stable/fast on dense, scaled features).
  - `max_iter`: increased to ensure convergence on our preprocessed feature set.

#### Validation & Reporting
For each model, I will:
1. Run `GridSearchCV` on **training data only** (`X_train_proc, y_train`) with `cv=5` and `scoring="accuracy"`.
2. Record the **best hyperparameters** and the corresponding **mean CV accuracy** (`best_score_`).
3. Evaluate the **refit** best model on **`X_test_proc, y_test`** to report **final test accuracy**.

This procedure prevents leakage from the test set, provides a robust in-sample model selection, and cleanly estimates generalization on the held-out test split.

In [31]:
# Cell 4: KNN and Logistic Regression Training

# Uses ONLY the training split from Cell 2 (X_train_proc, y_train) for tuning,
# then evaluates the refit best models on X_test_proc, y_test.

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score
)
import numpy as np
import pandas as pd

# --- Safety checks: ensure variables from Cell 2 exist ---
for var in ["X_train_proc", "X_test_proc", "y_train", "y_test"]:
    if var not in globals():
        raise RuntimeError(f"Variable '{var}' not found. Run Cell 2 first.")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# -------------------------
# KNN: grid + CV on training
# -------------------------
knn = KNeighborsClassifier()
knn_param_grid = {
    "n_neighbors": list(range(1, 32, 2)),   # 1..31 odd
    "weights": ["uniform", "distance"],
    "p": [1, 2],                            # Manhattan vs Euclidean
}

knn_search = GridSearchCV(
    estimator=knn,
    param_grid=knn_param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=False,
)

knn_search.fit(X_train_proc, y_train)

knn_best_params = knn_search.best_params_
knn_cv_acc = knn_search.best_score_

# Test-set predictions & probability for metrics
knn_best = knn_search.best_estimator_
knn_test_pred = knn_best.predict(X_test_proc)
# KNN supports predict_proba by default
knn_test_proba = knn_best.predict_proba(X_test_proc)[:, 1]

knn_test_acc = accuracy_score(y_test, knn_test_pred)
knn_bal_acc = balanced_accuracy_score(y_test, knn_test_pred)
knn_prec     = precision_score(y_test, knn_test_pred, zero_division=0)
knn_rec      = recall_score(y_test, knn_test_pred)
knn_f1       = f1_score(y_test, knn_test_pred)
knn_roc_auc  = roc_auc_score(y_test, knn_test_proba)
knn_pr_auc   = average_precision_score(y_test, knn_test_proba)

# --------------------------------
# Logistic Regression: grid + CV
# --------------------------------
lr = LogisticRegression(
    solver="lbfgs",
    penalty="l2",
    max_iter=2000,
    n_jobs=-1
)

lr_param_grid = {
    "C": [0.01, 0.1, 1, 10, 100]
}

lr_search = GridSearchCV(
    estimator=lr,
    param_grid=lr_param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=False,
)

lr_search.fit(X_train_proc, y_train)

lr_best_params = lr_search.best_params_
lr_cv_acc = lr_search.best_score_

lr_best = lr_search.best_estimator_
lr_test_pred = lr_best.predict(X_test_proc)
lr_test_proba = lr_best.predict_proba(X_test_proc)[:, 1]

lr_test_acc = accuracy_score(y_test, lr_test_pred)
lr_bal_acc  = balanced_accuracy_score(y_test, lr_test_pred)
lr_prec     = precision_score(y_test, lr_test_pred, zero_division=0)
lr_rec      = recall_score(y_test, lr_test_pred)
lr_f1       = f1_score(y_test, lr_test_pred)
lr_roc_auc  = roc_auc_score(y_test, lr_test_proba)
lr_pr_auc   = average_precision_score(y_test, lr_test_proba)

# -------------------------
# Clear, required reporting
# -------------------------
print("=== KNN (tuned) ===")
print(f"Best hyperparameters: {knn_best_params}")
print(f"Mean CV accuracy (best params): {knn_cv_acc:.4f}")
print(f"Test accuracy: {knn_test_acc:.4f}")
# Extra, imbalance-aware diagnostics
print(f"Balanced Accuracy: {knn_bal_acc:.4f} | Precision: {knn_prec:.4f} | Recall: {knn_rec:.4f} | F1: {knn_f1:.4f}")
print(f"ROC-AUC: {knn_roc_auc:.4f} | PR-AUC: {knn_pr_auc:.4f}")

print("\n=== Logistic Regression (tuned) ===")
print(f"Best hyperparameters: {lr_best_params}")
print(f"Mean CV accuracy (best params): {lr_cv_acc:.4f}")
print(f"Test accuracy: {lr_test_acc:.4f}")
# Extra, imbalance-aware diagnostics
print(f"Balanced Accuracy: {lr_bal_acc:.4f} | Precision: {lr_prec:.4f} | Recall: {lr_rec:.4f} | F1: {lr_f1:.4f}")
print(f"ROC-AUC: {lr_roc_auc:.4f} | PR-AUC: {lr_pr_auc:.4f}")

=== KNN (tuned) ===
Best hyperparameters: {'n_neighbors': 31, 'p': 1, 'weights': 'distance'}
Mean CV accuracy (best params): 0.9147
Test accuracy: 0.9169
Balanced Accuracy: 0.6207 | Precision: 0.7039 | Recall: 0.2528 | F1: 0.3720
ROC-AUC: 0.9210 | PR-AUC: 0.5510

=== Logistic Regression (tuned) ===
Best hyperparameters: {'C': 0.01}
Mean CV accuracy (best params): 0.9024
Test accuracy: 0.9022
Balanced Accuracy: 0.5151 | Precision: 0.4692 | Recall: 0.0345 | F1: 0.0643
ROC-AUC: 0.8826 | PR-AUC: 0.3493


## **Cell 5: Feature Pruning Plan**

**Goal.** Reduce the feature count to **at most half** of the original post-preprocessing features while keeping (or improving) generalization.

#### Methodology
I will apply a **filter-based statistical selection** on the **training split only** to avoid any test-set leakage:

1. **Compute Mutual Information (MI)** between each feature and the binary target on `X_train_proc, y_train`.  
   - MI captures **non-linear** associations and works well for mixed effects after our scaling/one-hot encoding.
2. **Rank features** by MI score in descending order.
3. **Select the top _K_ features**, where  
   \(K=\left\lfloor \frac{\text{original\_feature\_count}}{2} \right\rfloor\) (with a minimum of 1).  
   This guarantees the pruned set is **≤ 50%** of the original.
4. Apply the same column subset to `X_test_proc` for a fair held-out evaluation.
5. Re-run the **exact same GridSearchCV pipelines** from Cell 4 on the pruned matrices.

This approach is simple, fast, and model-agnostic; it gives a clean separation between selection (filter) and model training (wrapper).

#### Hypothesis
- **KNN** is distance-based and sensitive to noisy/irrelevant dimensions; pruning should **reduce variance** and often **improve** accuracy.  
- **Logistic Regression** already regularizes via L2, so gains may be **modest**; however, removing redundant one-hot columns can reduce multicollinearity and improve stability.  
Overall, I expect the **pruned set** to be **on par or slightly better**, especially for KNN.

In [32]:
# Cell 6: Training with Pruned Features

# ===== Cell 6: Training with Pruned Features =====
# Select top-k features (k = floor(n_features/2)) using Mutual Information on TRAIN ONLY,
# rebuild pruned X_train/X_test, then re-run tuned GridSearchCV pipelines (KNN, LR)
# including imbalance-aware metrics.

from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score
)
import numpy as np

# --- Safety checks: ensure variables from earlier cells exist ---
required_vars = ["X_train_proc", "X_test_proc", "y_train", "y_test", "feature_names"]
for var in required_vars:
    if var not in globals():
        raise RuntimeError(f"Variable '{var}' not found. Please run Cells 2 and 4 first.")

n_features = X_train_proc.shape[1]
k = max(1, n_features // 2)  # at most half (floor), minimum 1

# --- Compute Mutual Information on TRAIN ONLY ---
mi = mutual_info_classif(X_train_proc, y_train, random_state=42, discrete_features=False)
mi = np.nan_to_num(mi, nan=0.0)

# --- Get top-k indices & feature names ---
topk_idx = np.argsort(mi)[::-1][:k]
feature_names_pruned = [feature_names[i] for i in topk_idx]

# --- Build pruned matrices (same columns for train/test) ---
X_train_pruned = X_train_proc[:, topk_idx]
X_test_pruned  = X_test_proc[:, topk_idx]

print("Original feature count:", n_features)
print("Pruned feature count:", X_train_pruned.shape[1])
print("\nTop features (up to first 30):")
print(feature_names_pruned[:30])

# -------------------------
# Re-run tuned KNN on pruned set
# -------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

knn = KNeighborsClassifier()
knn_param_grid = {
    "n_neighbors": list(range(1, 32, 2)),   # 1..31 odd
    "weights": ["uniform", "distance"],
    "p": [1, 2],
}

knn_search_p = GridSearchCV(
    estimator=knn,
    param_grid=knn_param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=False,
)
knn_search_p.fit(X_train_pruned, y_train)

knn_best_params_p = knn_search_p.best_params_
knn_cv_acc_p = knn_search_p.best_score_

knn_best_p = knn_search_p.best_estimator_
knn_test_pred_p = knn_best_p.predict(X_test_pruned)
knn_test_proba_p = knn_best_p.predict_proba(X_test_pruned)[:, 1]

knn_test_acc_p = accuracy_score(y_test, knn_test_pred_p)
knn_bal_acc_p = balanced_accuracy_score(y_test, knn_test_pred_p)
knn_prec_p = precision_score(y_test, knn_test_pred_p, zero_division=0)
knn_rec_p = recall_score(y_test, knn_test_pred_p)
knn_f1_p = f1_score(y_test, knn_test_pred_p)
knn_roc_auc_p = roc_auc_score(y_test, knn_test_proba_p)
knn_pr_auc_p = average_precision_score(y_test, knn_test_proba_p)

# --------------------------------
# Re-run tuned Logistic Regression on pruned set
# --------------------------------
lr = LogisticRegression(
    solver="lbfgs",
    penalty="l2",
    max_iter=2000,
    n_jobs=-1
)
lr_param_grid = {
    "C": [0.01, 0.1, 1, 10, 100]
}
lr_search_p = GridSearchCV(
    estimator=lr,
    param_grid=lr_param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=False,
)
lr_search_p.fit(X_train_pruned, y_train)

lr_best_params_p = lr_search_p.best_params_
lr_cv_acc_p = lr_search_p.best_score_

lr_best_p = lr_search_p.best_estimator_
lr_test_pred_p = lr_best_p.predict(X_test_pruned)
lr_test_proba_p = lr_best_p.predict_proba(X_test_pruned)[:, 1]

lr_test_acc_p = accuracy_score(y_test, lr_test_pred_p)
lr_bal_acc_p = balanced_accuracy_score(y_test, lr_test_pred_p)
lr_prec_p = precision_score(y_test, lr_test_pred_p, zero_division=0)
lr_rec_p = recall_score(y_test, lr_test_pred_p)
lr_f1_p = f1_score(y_test, lr_test_pred_p)
lr_roc_auc_p = roc_auc_score(y_test, lr_test_proba_p)
lr_pr_auc_p = average_precision_score(y_test, lr_test_proba_p)

# -------------------------
# Required reporting (PRUNED FEATURE SET)
# -------------------------
print("\n=== [PRUNED] KNN (tuned) ===")
print(f"Best hyperparameters: {knn_best_params_p}")
print(f"Mean CV accuracy (best params): {knn_cv_acc_p:.4f}")
print(f"Test accuracy: {knn_test_acc_p:.4f}")
print(f"Balanced Accuracy: {knn_bal_acc_p:.4f} | Precision: {knn_prec_p:.4f} | Recall: {knn_rec_p:.4f} | F1: {knn_f1_p:.4f}")
print(f"ROC-AUC: {knn_roc_auc_p:.4f} | PR-AUC: {knn_pr_auc_p:.4f}")

print("\n=== [PRUNED] Logistic Regression (tuned) ===")
print(f"Best hyperparameters: {lr_best_params_p}")
print(f"Mean CV accuracy (best params): {lr_cv_acc_p:.4f}")
print(f"Test accuracy: {lr_test_acc_p:.4f}")
print(f"Balanced Accuracy: {lr_bal_acc_p:.4f} | Precision: {lr_prec_p:.4f} | Recall: {lr_rec_p:.4f} | F1: {lr_f1_p:.4f}")
print(f"ROC-AUC: {lr_roc_auc_p:.4f} | PR-AUC: {lr_pr_auc_p:.4f}")

# --- Analysis of Pruned Features (appended to Cell 6) ---
print("\n--- Analysis of Pruned Features ---")

# If baseline metrics from Cell 4 are available, print a direct comparison
baseline_vars_present = all(v in globals() for v in [
    "knn_test_acc", "knn_bal_acc", "lr_test_acc", "lr_bal_acc"
])

if baseline_vars_present:
    print(
        f"KNN: test accuracy {knn_test_acc:.4f} -> {knn_test_acc_p:.4f}; "
        f"balanced accuracy {knn_bal_acc:.4f} -> {knn_bal_acc_p:.4f}."
    )
    print(
        f"LR:  test accuracy {lr_test_acc:.4f} -> {lr_test_acc_p:.4f}; "
        f"balanced accuracy {lr_bal_acc:.4f} -> {lr_bal_acc_p:.4f}."
    )
else:
    print("Baseline (pre-pruning) metrics not found in scope; showing pruned-set metrics only above.")

print(
    "Interpretation: pruning reduced dimensionality to at most half the features. "
    "KNN may trade a small amount of overall accuracy for slightly improved class balance if "
    "informative but redundant features are removed. Logistic Regression tends to be stable "
    "if the dropped features were weak or collinear. Overall, pruning simplified the feature "
    "space without materially harming generalization."
)

Original feature count: 5
Pruned feature count: 2

Top features (up to first 30):
['absolute_magnitude', 'est_diameter_max']

=== [PRUNED] KNN (tuned) ===
Best hyperparameters: {'n_neighbors': 31, 'p': 1, 'weights': 'distance'}
Mean CV accuracy (best params): 0.9102
Test accuracy: 0.9101
Balanced Accuracy: 0.6217 | Precision: 0.5840 | Recall: 0.2636 | F1: 0.3632
ROC-AUC: 0.9148 | PR-AUC: 0.4911

=== [PRUNED] Logistic Regression (tuned) ===
Best hyperparameters: {'C': 0.01}
Mean CV accuracy (best params): 0.9027
Test accuracy: 0.9027
Balanced Accuracy: 0.5000 | Precision: 0.0000 | Recall: 0.0000 | F1: 0.0000
ROC-AUC: 0.8663 | PR-AUC: 0.2810

--- Analysis of Pruned Features ---
KNN: test accuracy 0.9169 -> 0.9101; balanced accuracy 0.6207 -> 0.6217.
LR:  test accuracy 0.9022 -> 0.9027; balanced accuracy 0.5151 -> 0.5000.
Interpretation: pruning reduced dimensionality to at most half the features. KNN may trade a small amount of overall accuracy for slightly improved class balance if info

## **Cell 7: "Choose Your Own Adventure" Plan**

#### Model Justification
I will use **XGBoost (XGBClassifier)** as the third model. Gradient-boosted trees typically outperform KNN and Logistic Regression on tabular data because they:
- Capture **nonlinear relationships** and complex feature interactions automatically.
- Are robust to feature scaling, missing values (after imputation), and mixed data types.
- Include built-in **regularization** (`reg_lambda`, `reg_alpha`) and shrinkage (`learning_rate`), which improve generalization.

Given the imbalanced nature of the target and the tabular structure of the data, XGBoost is well-suited to this task.  
To better reflect real-world performance, the **primary evaluation metrics** will be **Balanced Accuracy** and **F1-score**, as they account for class imbalance more effectively than raw accuracy.

#### Training Plan
- Use **`RandomizedSearchCV`** (3-fold **Stratified CV**, `scoring='accuracy'`) on the **training split only** (`X_train_proc, y_train`).
- Tune the main hyperparameters:
  - `n_estimators`, `max_depth`, `learning_rate`,
  - `subsample`, `colsample_bytree`,
  - `min_child_weight`, `reg_lambda`, `reg_alpha`.
- Handle class imbalance using **`scale_pos_weight = (#neg / #pos)`** computed from the training labels.
- After cross-validation, refit the best model on the **entire training set** and evaluate once on the **held-out test set**, reporting:
  - Accuracy, Balanced Accuracy, Precision, Recall, F1-score, ROC-AUC, and PR-AUC.
- The model will then be compared to KNN and Logistic Regression using these metrics, focusing on **Balanced Accuracy** and **F1-score** to determine the best-performing approach.

In [33]:
# Cell 8: "Choose Your Own Adventure" Implementation

# !pip -q install xgboost

import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_predict
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             precision_score, recall_score, roc_auc_score,
                             average_precision_score, precision_recall_curve)
from scipy.stats import randint, uniform
from numbers import Number
import math

# ---- Preconditions
for v in ["X_train_proc","X_test_proc","y_train","y_test"]:
    if v not in globals():
        raise RuntimeError(f"Missing {v}; run earlier cells first.")

# ---- Class imbalance handling
pos, neg = (y_train == 1).sum(), (y_train == 0).sum()
scale_pos_weight = neg / max(pos, 1)

# ---- Base model
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",     # prioritize minority-class performance
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
    scale_pos_weight=scale_pos_weight
)

# ---- Hyperparameter search space
param_dist = {
    "n_estimators": randint(400, 900),
    "learning_rate": uniform(0.02, 0.04),
    "max_depth": randint(3, 6),
    "min_child_weight": randint(1, 5),
    "subsample": uniform(0.8, 0.2),
    "colsample_bytree": uniform(0.7, 0.3),
    "reg_lambda": uniform(0.5, 1.5),
    "reg_alpha": uniform(0.0, 0.2),
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# ---- Tune for F1 to emphasize balance between precision and recall
search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=25,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    refit=True,
    random_state=42,
    verbose=0
)

search.fit(X_train_proc, y_train)
best = search.best_estimator_

# ---- Find probability threshold on TRAIN using CV predictions
train_proba_cv = cross_val_predict(
    best, X_train_proc, y_train, cv=cv, method="predict_proba", n_jobs=-1
)[:, 1]

prec, rec, thr = precision_recall_curve(y_train, train_proba_cv)
f1_vals = (2 * prec * rec) / np.clip(prec + rec, 1e-12, None)
best_idx = np.nanargmax(f1_vals)
best_thr = 0.5 if best_idx == len(thr) else thr[best_idx]

# ---- Refit and evaluate on TEST
best.fit(X_train_proc, y_train)
test_proba = best.predict_proba(X_test_proc)[:, 1]
y_pred = (test_proba >= best_thr).astype(int)

# ---- Metrics
acc  = accuracy_score(y_test, y_pred)
bacc = balanced_accuracy_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
pre  = precision_score(y_test, y_pred, zero_division=0)
rec_ = recall_score(y_test, y_pred)
roc  = roc_auc_score(y_test, test_proba)
ap   = average_precision_score(y_test, test_proba)

# ---- Clean parameter printout
def _to_builtin(x):
    if isinstance(x, Number):
        try:
            return float(x)
        except Exception:
            return x
    return x

best_params_clean = {k: round(_to_builtin(v), 4) for k, v in search.best_params_.items()}

# ---- Output summary (concise, no redundancy)
print("=== XGBoost (imbalance-aware) ===")
print(f"Best hyperparameters: {best_params_clean}")
print(f"CV best F1 (train CV): {search.best_score_:.4f}")
print(f"Decision threshold (from train CV): {best_thr:.3f}")
print(f"(train balance) pos={pos}, neg={neg}, scale_pos_weight={scale_pos_weight:.3f}")

print("\n--- Test metrics ---")
print(f"Accuracy:            {acc:.4f}")
print(f"Balanced Accuracy:   {bacc:.4f}")
print(f"Precision:           {pre:.4f}")
print(f"Recall:              {rec_:.4f}")
print(f"F1-score:            {f1:.4f}")
print(f"ROC-AUC:             {roc:.4f}")
print(f"PR-AUC (AvgPrec):    {ap:.4f}")

# ---- Clarification: why these metrics matter
print("\nNote:")
print("This is a classification task, so accuracy, F1, and AUC metrics are used (not R²).")
print("The dataset is imbalanced; 'scale_pos_weight' and threshold tuning prioritize recall and PR-AUC over raw accuracy.")
print("Accuracy may be slightly lower than simpler models, but minority-class performance is improved.")

# ---- Threshold sensitivity
y_pred_05 = (test_proba >= 0.50).astype(int)
acc_05 = accuracy_score(y_test, y_pred_05)
rec_05 = recall_score(y_test, y_pred_05)
print("\n--- Threshold analysis ---")
print(f"@0.50  → Accuracy={acc_05:.4f}, Recall={rec_05:.4f}")
print(f"@{best_thr:>4.2f} → Accuracy={acc:.4f}, Recall={rec_:.4f} (optimized for F1/PR trade-off)")

# =========================
# Comparison table (KNN vs LR vs XGBoost) — no hard-coded numbers
# =========================

def _fetch_metric_or_nan(var_names):
    """
    Try several possible variable names from earlier cells and
    return the first one found; otherwise return NaN.
    """
    for name in var_names:
        if name in globals():
            return globals()[name]
    return float("nan")

# KNN metrics (add your actual names first in each list)
knn_acc  = _fetch_metric_or_nan([
    "knn_test_acc", "knn_acc_test", "knn_accuracy_test", "KNN_test_accuracy"
])
knn_bacc = _fetch_metric_or_nan([
    "knn_bal_acc", "knn_test_bal_acc", "knn_balanced_accuracy_test", "KNN_test_balanced_accuracy"
])
knn_f1   = _fetch_metric_or_nan([
    "knn_f1", "knn_test_f1", "knn_f1_test", "KNN_test_f1"
])

# Logistic Regression metrics
lr_acc  = _fetch_metric_or_nan([
    "lr_test_acc", "logreg_test_acc", "lr_accuracy_test", "LR_test_accuracy"
])
lr_bacc = _fetch_metric_or_nan([
    "lr_bal_acc", "lr_test_bal_acc", "logreg_test_bal_acc", "lr_balanced_accuracy_test", "LR_test_balanced_accuracy"
])
lr_f1   = _fetch_metric_or_nan([
    "lr_f1", "lr_test_f1", "logreg_test_f1", "LR_test_f1"
])

# Build comparison DataFrame using variables only
comparison = pd.DataFrame({
    "Model": ["KNN", "Logistic Regression", "XGBoost"],
    "Accuracy": [knn_acc, lr_acc, acc],
    "Balanced Accuracy": [knn_bacc, lr_bacc, bacc],
    "F1-score": [knn_f1, lr_f1, f1]
})[["Model", "Accuracy", "Balanced Accuracy", "F1-score"]]

print("\n=== Model Comparison (primary metrics emphasized) ===")
display(comparison)

# Simple textual summary based on primary metrics (Balanced Accuracy, then F1)
import math
def _safe(val):  # treat NaN as very low
    return -1.0 if (val is None or (isinstance(val, float) and math.isnan(val))) else float(val)

bacc_values = {"KNN": _safe(knn_bacc), "Logistic Regression": _safe(lr_bacc), "XGBoost": _safe(bacc)}
f1_values   = {"KNN": _safe(knn_f1),  "Logistic Regression": _safe(lr_f1),  "XGBoost": _safe(f1)}

best_bacc_model = max(bacc_values, key=bacc_values.get)
best_f1_model   = max(f1_values,   key=f1_values.get)

print("\nSummary:")
print(f"- Best Balanced Accuracy: {best_bacc_model}")
print(f"- Best F1-score:          {best_f1_model}")

=== XGBoost (imbalance-aware) ===
Best hyperparameters: {'colsample_bytree': 0.7824, 'learning_rate': 0.0424, 'max_depth': 5.0, 'min_child_weight': 1.0, 'n_estimators': 830.0, 'reg_alpha': 0.1426, 'reg_lambda': 1.6412, 'subsample': 0.9123}
CV best F1 (train CV): 0.4871
Decision threshold (from train CV): 0.733
(train balance) pos=7072, neg=65596, scale_pos_weight=9.275

--- Test metrics ---
Accuracy:            0.8612
Balanced Accuracy:   0.8086
Precision:           0.3886
Recall:              0.7432
F1-score:            0.5104
ROC-AUC:             0.9244
PR-AUC (AvgPrec):    0.5573

Note:
This is a classification task, so accuracy, F1, and AUC metrics are used (not R²).
The dataset is imbalanced; 'scale_pos_weight' and threshold tuning prioritize recall and PR-AUC over raw accuracy.
Accuracy may be slightly lower than simpler models, but minority-class performance is improved.

--- Threshold analysis ---
@0.50  → Accuracy=0.8023, Recall=0.9717
@0.73 → Accuracy=0.8612, Recall=0.7432 (o

,Model,Accuracy,Balanced Accuracy,F1-score
0,KNN,0.916942,0.620682,0.372035
1,Logistic Regression,0.902246,0.515147,0.064278
2,XGBoost,0.861240,0.808588,0.510390



Summary:
- Best Balanced Accuracy: XGBoost
- Best F1-score:          XGBoost
